In [ ]:
import numpy as np
import torch

In [ ]:
from pid.utils import generate_randomBC

dm = 2  # Number of dimensions of M
dx = 100
dy = 70

# set manual seed
torch.manual_seed(0)
np.random.seed(0)

hx, hy = generate_randomBC(dm, dx, dy)

sigm = np.eye(dm)
sigx_m = np.eye(dx)
sigy_m = np.eye(dy)
sigw = np.zeros((dx, dy))

# Covariance matrix construction for both unique or redundant
cov = np.block([[sigm, sigm @ hx.T, sigm @ hy.T],
                [hx @ sigm, hx @ sigm @ hx.T + sigx_m, hx @ sigm @ hy.T + sigw],
                [hy @ sigm, hy @ sigm @ hx.T + sigw.T, hy @ sigm @ hy.T + sigy_m]])

print(cov.shape)

In [ ]:
n = 100000   # number of samples

mean = np.zeros(dm+dx+dy)
mxy = np.random.multivariate_normal(mean, cov, size=n)
cov = np.corrcoef(mxy.T)

m_data, x_data, y_data = np.hsplit(mxy, [dm, dm+dx])  # split m, x, y
print(m_data.shape, x_data.shape, y_data.shape)

new_mxy = np.hstack((m_data, x_data, y_data))
new_cov = np.corrcoef(new_mxy.T)

def standardize_data(data):
    data = torch.from_numpy(np.nan_to_num((data - data.mean(axis=0)) / data.std(axis=0))).float()
    data = torch.clip(data, min=-10, max=10)
    return data

m_standardized = standardize_data(m_data)
x_standardized = standardize_data(x_data)
y_standardized = standardize_data(y_data)

dataset = torch.utils.data.TensorDataset(x_standardized, y_standardized, m_standardized)
train_loader = torch.utils.data.DataLoader(dataset, batch_size=512, shuffle=True)

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm

def train_lmi(model, train_loader, lr=0.0001, epochs=300, verbose=False):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, eps=1e-07)

    ############# training loop #############
    losses = []
    for epoch in tqdm(range(epochs)):
        epoch_losses = []
        for batch_n, (x_batch, y_batch, m_batch) in enumerate(train_loader):
            x_batch, y_batch, m_batch = x_batch.to(device), y_batch.to(device), m_batch.to(device)

            model.train()
            optimizer.zero_grad()

            loss = model.learning_loss(x_batch, y_batch, m_batch)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        avg_loss = sum(epoch_losses) / len(epoch_losses)
        losses.append(avg_loss)

    if verbose:
        plt.figure(figsize=(10, 5))
        plt.plot(losses)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training Loss')
        plt.show()

    return model

In [ ]:
import pid.models.lmi as lmi

enable_cuda=True
device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')
print(f"Using device: {device}")

latent_dim = 16

encoder = lmi.AEMINE(dx, dy, dm, latent_dim, alpha=1.0, lam=1.0).to(device)
lmi_encoder = train_lmi(encoder, train_loader, lr=2e-4, epochs=2000, verbose=True)

In [ ]:
# Use the trained encoder to get the latent representations
x_latent = []
y_latent = []
m_latent = []
for x_batch, y_batch, m_batch in train_loader:
    lmi_encoder.eval()
    x_batch, y_batch, m_batch = x_batch.to(device), y_batch.to(device), m_batch.to(device)

    with torch.no_grad():
        Z_x, Z_y = lmi_encoder.encode(x_batch, y_batch)

    x_latent.append(Z_x)
    y_latent.append(Z_y)
    m_latent.append(m_batch)

x_latent = torch.cat(x_latent)
y_latent = torch.cat(y_latent)
m_latent = torch.cat(m_latent)

new_mxy = np.hstack((m_latent.cpu().numpy(), x_latent.cpu().numpy(), y_latent.cpu().numpy()))
new_cov = np.corrcoef(new_mxy.T)
print(new_cov.shape)

In [ ]:
from pid.optimizers import exact_gauss_tilde_pid, exact_gauss_thin_pid

ret = exact_gauss_tilde_pid(cov, dm, dx, dy)
imxy, uix, uiy, ri, si = ret[2], *ret[-4:]
print(f"Truth, Tilde-PID: R: {ri}, U1: {uix}, U2: {uiy}, S: {si}")

ret = exact_gauss_tilde_pid(new_cov, dm, latent_dim, latent_dim)
imxy, uix, uiy, ri, si = ret[2], *ret[-4:]
print(f"Latent MI, Tilde-PID: R: {ri}, U1: {uix}, U2: {uiy}, S: {si}")

ret = exact_gauss_thin_pid(cov, dm, dx, dy)
imxy, uix, uiy, ri, si = ret[2], *ret[-4:]
print(f"Truth, Thin-PID: R: {ri}, U1: {uix}, U2: {uiy}, S: {si}")

ret = exact_gauss_thin_pid(new_cov, dm, latent_dim, latent_dim)
imxy, uix, uiy, ri, si = ret[2], *ret[-4:]
print(f"Latent MI, Thin-PID: R: {ri}, U1: {uix}, U2: {uiy}, S: {si}")